In [13]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial.distance import pdist

from dtaidistance import dtw

## Load Generated Samples


In [ ]:
INPUT_PATH = Path('../../data/generated/generated_returns.json')

with open(INPUT_PATH) as f:
    data = json.load(f)

conditions = data['conditions']
samples_np = np.array(data['samples'], dtype=np.float64)  # (N, T)

N, T = samples_np.shape
n_pairs = N * (N - 1) // 2

print("Conditions:")
for k, v in conditions.items():
    print(f"  {k}: {v}")
print(f"\nSamples: {N} sequences of length {T}")
print(f"Pairs to evaluate: {n_pairs:,}")

Conditions:
  trend: 0.0
  realized_vol: 50.0
  interest_rate: 5.0
  volatility_index: 20.0
  guidance_scale: 1.0

Samples: 200 sequences of length 64
Pairs to evaluate: 19,900


## Euclidean Distances


In [15]:
print("Computing pairwise Euclidean distances...")
euc_dists = pdist(samples_np, metric='euclidean')  # shape (n_pairs,)
print(f"Done. Shape: {euc_dists.shape}")

Computing pairwise Euclidean distances...
Done. Shape: (19900,)


## DTW Distances


In [16]:
# distance_matrix_fast needs the C extension; use_c=False uses pure Python (slower but works everywhere).
print("Computing pairwise DTW distances (slower without C library; progress below)...")
dtw_matrix = dtw.distance_matrix(
    samples_np,
    use_c=False,
    parallel=True,
    show_progress=True,
)
dtw_dists = dtw_matrix[np.triu_indices(N, k=1)]
print(f"Done. Shape: {dtw_dists.shape}")

Computing pairwise DTW distances (slower without C library; progress below)...
Done. Shape: (19900,)


## Summary Statistics


In [17]:
cond_str = f"trend={conditions['trend']}, realized_vol={conditions['realized_vol']}"

print(f"Condition: {cond_str}")
print(f"{'Metric':<12} {'Mean':>10} {'Std':>10} {'Median':>10} {'Min':>10} {'Max':>10}")
print("-" * 62)
for name, dists in [("Euclidean", euc_dists), ("DTW", dtw_dists)]:
    print(
        f"{name:<12} "
        f"{dists.mean():>10.4f} "
        f"{dists.std():>10.4f} "
        f"{np.median(dists):>10.4f} "
        f"{dists.min():>10.4f} "
        f"{dists.max():>10.4f}"
    )

Condition: trend=0.0, realized_vol=50.0
Metric             Mean        Std     Median        Min        Max
--------------------------------------------------------------
Euclidean       10.6532     0.9693    10.6404     6.6253    16.5504
DTW              6.0297     0.6060     5.9646     4.2695    10.1415
